#Aplicando LLM com JSON

Chaves e pacotes

In [10]:
!pip install llama-index -q
!pip install llama-index-llms-groq -q
!pip install -q llama-index-embeddings-huggingface
!pip install llama-index-vector-stores-chroma -q
!pip install -q gradio
!pip install llama-index-readers-file -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.7 MB/s eta 0:00:00


In [19]:
import pandas as pd
import chromadb
import time
import gradio as gr
import json
from google.colab import userdata
from llama_index.llms.groq import Groq
from llama_index.core.chat_engine import SimpleChatEngine
from llama_index.core import StorageContext
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex
from llama_index.core import load_index_from_storage
from llama_index.core import get_response_synthesizer
from llama_index.core import PromptTemplate
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.response_synthesizers import BaseSynthesizer
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import Settings

In [2]:
# Chave Groq salva no Colab
GROQ_API = userdata.get('GROQ_API')

# Modelo de LLM - Groq OpenAI
llm = Groq(model="openai/gpt-oss-20b", api_key=GROQ_API, max_tokens=4096)

url= "https://raw.githubusercontent.com/PRBerlanda/LlamaIndex/refs/heads/Automated-Response-JSON-Formatting/livros_mais_vendidos.csv"
df = pd.read_csv(url)

df.to_csv("csv/livros_mais_vendidos.csv", index=False)

In [3]:
# Chat básico
chat_engine = SimpleChatEngine.from_defaults(llm=llm)
# chat_engine.chat_repl()

Pergunta e resposta

In [4]:
resposta = llm.complete("Recomende livros de romance")

In [5]:
print(resposta.text)

Aqui vai uma seleção de romances que abrange vários subgêneros e épocas. Todos os títulos estão disponíveis em português (ou têm tradução oficial) e são bem‑recebidos por leitores de todo o mundo.  

| # | Título | Autor | Subgênero | Breve sinopse | Por que vale a pena ler |
|---|--------|-------|-----------|---------------|------------------------|
| 1 | **Romeu e Julieta** | William Shakespeare | Drama clássico | A tragédia de dois jovens apaixonados de famílias rivais em Verona. | É a obra‑fundação do romance, com linguagem poética e temas universais. |
| 2 | **Orgulho e Preconceito** | Jane Austen | Romance clássico | Elizabeth Bennet e Mr. Darcy navegam entre orgulho, preconceito e amor em 19º c. Inglaterra. | Humor, crítica social e um final feliz que ainda inspira adaptações. |
| 3 | **O Morro dos Ventos Uivantes** | Emily Brontë | Romance gótico | A paixão turbulenta entre Heathcliff e Catherine em um cenário de tormentas e vingança. | Intensidade emocional e atmosfera sombria

#Criando uma base vetorial

###Embedding dos dados

In [6]:
# Class Wrapper personalizado para compatibilizar com a assinatura Chroma
class ChromaEmbeddingWrapper:
    def __init__(self, model_name):
        self.model = HuggingFaceEmbedding(model_name=model_name)

    def __call__(self, input):
        # Converte o input a um formato compatível com HuggingFaceEmbedding
        return self.model.embed(input)

# Inicializa o Wrapper personalizado
embed_model_chroma = ChromaEmbeddingWrapper("intfloat/multilingual-e5-large")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

###Base de dados vetorial

In [7]:
# Inicializar o cliente persistente
db = chromadb.PersistentClient(path="./chroma_db")

# Atribuindo o cliente a uma variável para uso posterior
chroma_client = db

# Nome da coleção
collection_name = "meus_documentos"

# Carregar a coleção, se existir; caso contrário, criá-la
try:
    # Use `get_or_create_collection` para evitar recriar a coleção
    chroma_collection = chroma_client.get_or_create_collection(
        name=collection_name
    )
except Exception as e:
    print(f"Erro ao carregar ou criar a coleção: {e}")

###Montando pipeline

In [20]:
# Criando uma instância do ChromaVectorStore para armazenamento de vetores
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
# Configurando um contexto de armazenamento com o vector_store como armazenamento de vetores
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Inserir o diretório onde estão os arquivos, carregando os documentos.
documents = SimpleDirectoryReader("csv").load_data()

# Modelo de embedding da huggingface. Usado para converter texto em vetores
# numéricos (embeddings). Este modelo funciona bem em português.
embed_model = HuggingFaceEmbedding(model_name="intfloat/multilingual-e5-large")

# Configure LlamaIndex global settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512  # Reduce chunk size to fit within LLM context window
Settings.chunk_overlap = 20

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [21]:
start_time = time.time()

index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context, embed_model=embed_model
)

end_time = time.time()
execution_time = end_time - start_time

print(f"Tempo de execução: {execution_time} segundos")

Tempo de execução: 691.964031457901 segundos


In [22]:
# Obter IDs de todos os documentos
doc_ids = chroma_collection.get()['ids']
print(f"Total de IDs: {len(doc_ids)}")

index = load_index_from_storage(storage_context, embed_model=embed_model)

query_engine = index.as_query_engine(similarity_top_k=1, llm=llm)

Total de IDs: 240


Inserindo dados novos aos já existentes

In [23]:
# Carregando dados de arquivos não estruturados
novos_documentos = SimpleDirectoryReader("artigos").load_data()

# Após carregar, insere-se os dados no banco vetorial para aumentar
# a carga de contexto da LLM.
for doc in novos_documentos:
    index.insert(doc)

###TopK resultados

In [26]:
query_str = "Livros de romance"

In [27]:
query_embeded = embed_model.get_text_embedding(query_str)

In [28]:
chroma_collection.query(query_embeded, n_results=10, include=['distances', 'embeddings'])

{'ids': [['40b4fc25-afbb-4889-ba11-3de8844d7318',
   'c48e53b1-ef81-41bc-8c31-a21213d97019',
   '14c936de-e067-42a3-b1c3-919d8b4f24cc',
   '978e99d9-2291-47a8-beaa-21298596fc85',
   '20576290-7ba0-4d03-888a-816ffc2eae2c',
   '6536985b-7b94-4fd4-877d-32f00bec7e0e',
   '61a47336-f4e2-4246-b3a9-db1a40253a27',
   '881907ef-14d2-4797-83db-9f0751bee03e',
   'c7e2b3ab-f3e4-4043-b03b-b14208f46c13',
   'f6277cd3-0111-477d-b534-0a64e32d35d2']],
 'embeddings': [array([[ 0.02034746,  0.00922666, -0.02066277, ...,  0.00392685,
          -0.0017501 ,  0.03043894],
         [ 0.02573272, -0.0077506 , -0.00574945, ..., -0.00679026,
           0.00214347,  0.03426651],
         [ 0.00515677, -0.01434251, -0.0066998 , ..., -0.01204628,
          -0.01792503,  0.01538347],
         ...,
         [ 0.00742915, -0.00693728, -0.00488355, ..., -0.00854556,
          -0.02482307,  0.00396573],
         [ 0.00595423,  0.01705809, -0.00697422, ...,  0.00351386,
          -0.03390727,  0.0013693 ],
         [ 0.

In [29]:
# Obter IDs de todos os documentos
doc_ids = chroma_collection.get()['ids']
print(f"Total de IDs: {len(doc_ids)}")
print("Amostra de IDs:", doc_ids[:5])  # Exibe os primeiros 5 IDs para inspecionar

# Extrair uma amostra dos documentos
amostra = chroma_collection.get([doc_ids[i] for i in range(5)])  # Ajuste o número se quiser mais amostras
print("Amostra de documentos:", amostra)

Total de IDs: 295
Amostra de IDs: ['d295a00e-2d2a-4843-b013-6920bad69a17', '1b72ab49-d152-454e-80b9-073c226b446b', 'b203298e-d2fd-418c-b837-716559e8c2e7', '40b4fc25-afbb-4889-ba11-3de8844d7318', '181429b6-2030-43f8-802e-a05fb64e0463']
Amostra de documentos: {'ids': ['d295a00e-2d2a-4843-b013-6920bad69a17', '1b72ab49-d152-454e-80b9-073c226b446b', 'b203298e-d2fd-418c-b837-716559e8c2e7', '40b4fc25-afbb-4889-ba11-3de8844d7318', '181429b6-2030-43f8-802e-a05fb64e0463'], 'embeddings': None, 'documents': ['É ASSIM QUE COMEÇA (EDIÇÃO DE COLECIONADOR), Romances, Todo final tem um começo. Agora em uma edição especial de colecionador, com pintura trilateral e exclusiva para o Brasil, conheça a sequência de \n                        É assim que acaba\n                        . Da autora best-seller mundial e fenômeno editorial Colleen Hoover, que já vendeu mais de dois milhões de livros no país.\n                    \n                \n                \n                \xa0\n                \n      

In [30]:
chroma_collection.get("c20e8f26-e064-4464-8718-bef53f762f57")

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

### Inserindo nova fonte de dados não estruturada



###Consulta multidocumento

In [33]:
retriever = index.as_retriever()

synthesizer = get_response_synthesizer(response_mode="compact", llm=llm)

qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Responda em português.\n"
    "GERE APENAS um objeto JSON válido, e NADA MAIS. Sem texto adicional, explicações ou conversa.\n"
    "O formato JSON DEVE ser exatamente este:\n"
    "{\n"
    "  \"data\":{\n"
    "    \"livros\": [\n"
    "      {\n"
    "        \"titulo\": \"Titulo do livro\",\n"
    "        \"categoria\": \"Categoria do livro\",\n"
    "        \"Justificativa\": \"Porque recomenda\"\n"
    "      }\n"
    "    ]\n"
    "  }\n"
    "}\n"
    "Só recomende livros dessa base de dados para a query: {query_str}\n"
)

class RAGStringQueryEngine(CustomQueryEngine):

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: Groq
    qa_prompt: PromptTemplate

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)

        context_str = "\n\n".join([n.node.get_content() for n in nodes])

        response = self.llm.complete(
            self.qa_prompt.format(context_str=context_str, query_str=query_str)
        )

        return response.text

query_engine_customize = RAGStringQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=qa_prompt
)


resposta = query_engine_customize.query("Quais livros de Romance").response
resposta_json = json.loads(resposta)

for livro in resposta_json["data"]["livros"]:
  titulo = livro.get("titulo").upper()

  if df.query("nome == @titulo").empty:
    print(f"Livro {titulo} não encontrado")

# Pipeline e publicação MVP



####Preparando pipeline

In [34]:
# Valida os dados verificando se estão presentes na base de daos
def validando(resposta_json):
  for livro in resposta_json["data"]["livros"]:
    titulo = livro.get("titulo").upper()

  if df.query("nome.str.contains(@titulo)").empty:
    raise Exception(f"Livro {titulo} não encontrado")


# Melhorando a resposta do usuário ao integrar a response com consulta
def formatando_respostas(response, consulta):
  prompt = f'''
  Formate a resposta para o usuario
  Sua persona é um atendente da empresa buscante que vai recomendar livros usando a consulta do usuario como referencia
  E recomendar somente esses livros que estão em response
  consulta: {consulta}
  response: {response}
  '''
  resposta_final = llm.complete(prompt=prompt).text
  return resposta_final


def pipeline_completo(consulta): #Receberá a entrada do usuário
    # Realizando a consulta do usuário
    response = query_engine_customize.query(consulta).response
    # Transformando em JSON
    resposta_json = json.loads(response)

    validando(resposta_json)

    # Formatar a resposta
    response = formatando_respostas(response, consulta)

    return str(response)

In [35]:
print(pipeline_completo("Quais livros de Romance"))

Olá! 👋  
Aqui na Buscante, temos uma seleção incrível de romances que vão fazer seu coração bater mais forte. Confira as opções que preparamos especialmente para você:

| # | Título | Justificativa |
|---|--------|---------------|
| 1 | **É ASSIM QUE COMEÇA (EDIÇÃO DE COLECIONADOR)** | Clássico tradicional que oferece uma história de amor envolvente e profunda. |
| 2 | **TUDO É RIO** | História leve e divertida que conquista o coração com personagens cativantes. |
| 3 | **TÁTICAS DO AMOR** | Narrativa leve e divertida, perfeita para quem busca romance com humor e charme. |
| 4 | **NEM TE CONTO** | Romance intenso e emocionante, ideal para leitores que apreciam drama e paixão. |
| 5 | **O BEIJO DA NEVE** | Histórias envolventes que misturam romance com elementos de mistério e aventura. |
| 6 | **EU E ESSE MEU CORAÇÃO** | Romance intenso que explora emoções profundas e relações complexas. |
| 7 | **A MANDÍBULA DE CAIM** | Aborda temas complexos como amor e sacrifício, oferecendo uma tram

#### Subindo aplicação Gradio

In [36]:
demo = gr.Interface(
    fn=pipeline_completo,
    inputs=["text"],
    outputs=["text"],
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://97dc43900c6c80699d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://97dc43900c6c80699d.gradio.live


## Configurando no Streamlit

In [ ]:
!pip install streamlit -q

In [ ]:

# Funções do pipeline
def validando(resposta_json):
    for livro in resposta_json['data']['livros']:
        titulo = livro['titulo'].upper()
        if df.query("nome == @titulo").empty:
            raise Exception(f"Livro {titulo} não encontrado")

def formatando_respostas(response, consulta):
    prompt = f'''
    Formate a resposta para o usuario
    Sua persona é um atendente da empresa buscante que vai recomendar livros usando a consulta do usuario como referencia
    E recomendar somente esses livros que estão em response
    consulta: {consulta}
    response: {response}
    '''
    resposta_final = llm.complete(prompt=prompt).text
    return resposta_final

def pipeline_completo(consulta):
    response = query_engine_customize.query(consulta).response
    resposta_json = json.loads(response)
    validando(resposta_json)
    response = formatando_respostas(response, consulta)
    return str(response)

# Interface Streamlit
st.title("Especialista BotBuscante")
consulta = st.text_input("Digite sua consulta:")
if st.button("Buscar"):
    try:
        resposta = pipeline_completo(consulta)
        st.success("Recomendações:")
        st.write(resposta)
    except Exception as e:
        st.error(f"Erro: {str(e)}")

# Executar streamlit run app.py no terminal

Extra: Buscando dados em chunks

In [ ]:
chunk_size = 10
chunks = [df.iloc[i:i + chunk_size] for i in range(0, len(df), chunk_size)]

summarizing_prompt = """
Resuma o seguinte conjunto de dados de livros em uma descrição breve que destaque os gêneros principais:
Conjunto de dados:
"""
resumos = []

for i, chunk in enumerate(chunks):
    prompt = summarizing_prompt + "\n".join(chunk['nome'].values)
    resposta = llm.complete(prompt=prompt).text.strip()
    resumos.append({"chunk_index": i, "resumo": resposta})

relevance_prompt = """
Com base nos resumos abaixo, selecione os chunks mais relevantes para "Recomende livros de romance".
Retorne a resposta exclusivamente no seguinte formato JSON (não inclua explicações adicionais ou comentários):

{
"relevantes": [índices]
}

Os índices devem corresponder aos números dos chunks mencionados nos resumos.
"""

resumo_texto = "\n".join([f"Chunk {r['chunk_index']}: {r['resumo']}" for r in resumos])
relevantes_resposta = llm.complete(prompt=relevance_prompt + resumo_texto).text.strip()
relevantes_json = json.loads(relevantes_resposta)
chunks_relevantes = [chunks[i] for i in relevantes_json['relevantes']]

recomendacao_prompt = """
Recomende livros de romance
Formate em JSON
Responda em JSON
Não escreva mais nada:
{
  "data": {
    "livros": [
      {
        "titulo": "Titulo do livro",
        "categoria": "Categoria do livro",
        "justificativa": "Por que recomenda"
      }
    ]
  }
}
"""

resultados = []

for chunk in chunks_relevantes:
    prompt = recomendacao_prompt + "\n".join(chunk['nome'].values)
    resposta = llm.complete(prompt=prompt).text.strip()
    try:
        resposta_json = json.loads(resposta)
        resultados.extend(resposta_json['data']['livros'])
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError: {e}")
        print(f"LLM response that caused the error: {resposta}")
        continue
    except KeyError as e:
        print(f"KeyError accessing JSON structure: {e}")
        print(f"LLM response that caused the error: {resposta}")
        continue